In [ ]:
import os
from glob import glob
import re
import json

# output_dir = os.path.join('outputs','splits')
output_dir = os.path.join('outputs','splits')

# font_fname = os.environ.get('PROJECT_FONT_PATH', '') # point at any local CJK/serif .ttf
font_fname = os.environ.get('PROJECT_FONT_PATH', '') # point at any local CJK/serif .ttf

import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
from matplotlib import font_manager as fm
import seaborn as sbn

from src.utils.preprocessing import Preprocessor



negative_patterns = ['without','not? (sugges|compat)', 'unlike', 'neither', 'no', 'negative']
uncertain_patterns = ['rule out', 'Rule Out', "Rule out"]

def negative_words():
    return ' ?no | ?neither | ?unlike | ?without | ?not? (sugges|compat)| ?negative | ?previous'

def uncertain_words():
    return ' ?rule out '

def check_label(df, s):
    df = df.to_frame()
    df['label'] = s
    return df


font_family = fm.FontProperties(fname=font_fname).get_name() #폰트 설정
plt.rcParams["font.family"] = font_family  #폰트 적용

data_dir_path = glob(os.path.join('data','reviewed_labels','*'))


# figdir = os.path.join(output_dir,'figures')
# filedir = os.path.join(output_dir,'files')
# os.makedirs(figdir, exist_ok=True)
# os.makedirs(filedir, exist_ok=True)
# pd.read_excel(data_dir_path[5], engine = 'openpyxl')

In [ ]:
def excel_loader(paths, target):
    paths = [p for p in paths if target in p]
    df = pd.concat(
        [column_selector(
            pd.read_excel(p, engine = 'openpyxl')).dropna().set_axis(['index','검사결과결론내용', 'label'], axis = 1) for p in paths])
    df = data_label_prepr(df)
    return df


def column_selector(df, fixed_col = ['index','검사결과결론내용']):
    # print([c for c in df.columns if c not in fixed_col][-1])
    temp = df.loc[:, fixed_col + [[c for c in df.columns if c not in fixed_col][-1]]]
    print(temp.columns)
    return temp


def data_label_prepr(df):
    df['label'] = df['label'].apply(label_repr)
    return df


def label_repr(s):
    if 'recur' in s:
        return 'positive'
    elif 'region' in s:
        return 'RLN'
    elif 'meta' in s:
        return 'positive'
    elif 'positive' in s:
        return 'positive'
    elif 'x' in s:
        return 'negative'
    elif 'negative' in s:
        return 'negative'
    elif 'uncertain' in s:
        return 'uncertain'
    else:
        assert False, s

In [ ]:
ldf = excel_loader(data_dir_path, 'recur')

In [ ]:
ldf.drop_duplicates(['검사결과결론내용']).label.value_counts()

In [ ]:
ldf

In [ ]:
raw_dir_path = sorted(glob(os.path.join('data','reports','*')))
print(raw_dir_path)


In [ ]:
df = pd.read_csv(raw_dir_path[0], encoding = 'utf-8-sig', engine = 'c')
text_df = df['검사결과결론내용'].dropna()

preprocessor = Preprocessor(
    df = text_df,
    negative_patterns = negative_patterns,
    uncertain_patterns = uncertain_patterns
)


In [ ]:
recur_dict = preprocessor.target_filtering('recur')
metas_dict = preprocessor.target_filtering('metas')

merged_recur_df = pd.concat([check_label(d, k) for k, d in recur_dict.items()]).sort_index()
merged_metas_df = pd.concat([check_label(d, k) for k, d in metas_dict.items()]).sort_index()

df.loc[merged_recur_df.index, 'recur_text'] = merged_recur_df['검사결과결론내용']
df.loc[merged_recur_df.index, 'recur_label'] = merged_recur_df['label']
df.loc[merged_metas_df.index, 'metas_text'] = merged_metas_df['검사결과결론내용']
df.loc[merged_metas_df.index, 'metas_label'] = merged_metas_df['label']

In [ ]:
import numpy as np

from matplotlib import pyplot as plt
import seaborn as sbn

from sklearn.ensemble import RandomForestClassifier
from sklearn.manifold import Isomap, TSNE
from sklearn.cluster import DBSCAN
from umap import UMAP
from sklearn.metrics import confusion_matrix, classification_report

from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import torch

from transformers import AutoModel, AutoTokenizer



In [ ]:

class TextDataset(Dataset):
    def __init__(self, text_series):
        self.text_series = text_series.tolist()

    def __len__(self):
        return len(self.text_series)

    def __getitem__(self, idx):
        return self.text_series[idx]
        
        

@torch.no_grad()
def llm_embedding_fn(text_df):

    language_model.eval()
    language_model.cuda()

    dl = DataLoader(
        TextDataset(text_df),
        batch_size = 128,
        shuffle = False
    )

    vector_outputs = []
    # ids_outputs = []
    for text in tqdm(dl):
        token = tokenizer(
            text,
            # [tokenizer.special_tokens_map['cls_token']+t+tokenizer.special_tokens_map['sep_token'] for t in text], 
            return_tensors='pt', padding=True, truncation=True, padding_side='right'
            )
        # input_ids = token['input_ids']
        # with torch.no_grad():
        output = language_model(**{k:v.cuda() for k , v in token.items()}).last_hidden_state[:,0].cpu().float().numpy()
        
        vector_outputs.append(output)
    # ids_outputs.append(input_ids.numpy())

    return vector_outputs


In [ ]:

model_path = os.path.join('..','..','..','model','PubMedBERT-base-uncased-sts-combined')

tokenizer = AutoTokenizer.from_pretrained(model_path)
language_model = AutoModel.from_pretrained(
        model_path,
        dtype="bfloat16",
    )

txt2label = {
    'meta' : 'positive',
    'regional LN metastasis' : 'negative',
    'recur' : 'positive',
    'uncertain' : 'uncertain',
}
textmapping_fn = lambda x: txt2label[x] if x in txt2label else 'negative'


Note: consolidating the multiple human-review rounds into the reviewed labels is a study-design step described in the paper; here we treat the consolidated reviewed labels as a given input. The preprocessing -> embedding -> clustering weak-negative extraction below is fully reproducible.

In [ ]:
# meta_positive, recur_positive = sorted(glob(os.path.join('data','reviewer','*positive_encoding.csv')))[:2]

reviewed_meta_df = excel_loader(data_dir_path, 'metas')
reviewed_recur_df = excel_loader(data_dir_path, 'recur')


In [ ]:
# merged_df = merged_metas_df.drop_duplicates(subset=['검사결과결론내용'])
merged_df = merged_recur_df
reviewed_df = reviewed_recur_df

idx = reviewed_df.loc[:, 'index']
locs = merged_df['검사결과결론내용'].isin(reviewed_df['검사결과결론내용'].drop_duplicates().unique())
print(merged_df.loc[locs, 'label'].value_counts())
merged_df = merged_df.loc[~locs]
print(merged_df['label'].value_counts())

In [ ]:
vector_outputs = llm_embedding_fn(merged_df['검사결과결론내용'])
numpy_outputs = np.concatenate([i for i in vector_outputs], axis = 0)

In [ ]:
manifold = UMAP()
manifold_eos_vectors = manifold.fit_transform(numpy_outputs)

In [ ]:
# db = DBSCAN(eps = 1) # recur
db = DBSCAN(eps = 1) # metas
numb_cluster = db.fit_predict(numpy_outputs)

In [ ]:
# sub_manifold_vectors = manifold_eos_vectors[:part_df.shape[0], :]
# sub_numb_cluster = numb_cluster[:part_df.shape[0]]

sub_manifold_vectors = manifold_eos_vectors
sub_numb_cluster = numb_cluster

plot_df = pd.DataFrame(
    np.concatenate([sub_manifold_vectors, sub_numb_cluster[:, None]], axis = 1),
    columns = ['x1','x2','c']
)

sbn.jointplot(
    data = plot_df, x = 'x1', y = 'x2', 
    hue = 'c',
    joint_kws = {'s' : 2, 'alpha' : .5},
    )

In [ ]:
merged_df.shape, numb_cluster.shape

In [ ]:
temp_df = merged_df.copy()
temp_df['c'] = numb_cluster
reviewed_negs = reviewed_df.query('label == "negative"').drop_duplicates(subset=['검사결과결론내용'])['검사결과결론내용'].to_list()

In [ ]:
neg_cs = []
other_cs = []
for c, sdf in temp_df.groupby('c'):
    # print(sdf.label.unique())
    if 'positive' in sdf.label.unique():
        other_cs.append([c, sdf.shape[0], sdf.drop_duplicates(subset=['검사결과결론내용']).shape[0]])
        continue
    if 'uncertain' in sdf.label.unique():
        other_cs.append([c, sdf.shape[0], sdf.drop_duplicates(subset=['검사결과결론내용']).shape[0]])
        continue
    # print(sdf.label.unique())
    # else:
    if len(sdf.label.unique()) == 1:
        if sum([i in reviewed_negs for i in sdf['검사결과결론내용'].drop_duplicates().unique()]) == 0:
            neg_cs.append([c, sdf.shape[0], sdf.drop_duplicates(subset=['검사결과결론내용']).shape[0]])

In [ ]:
sorted_c = sorted(neg_cs, key=lambda x: x[2], reverse=True)
sorted_c = np.c_[
    sorted_c, 
    np.array(sorted_c)[:,-1].cumsum() / np.array(sorted_c)[:,-1].sum() * 100,
    np.array(sorted_c)[:, 1].cumsum() / np.array(sorted_c)[:,1].sum() * 100,
]

In [ ]:
thr = 50

fig, ax = plt.subplots()
tax = ax.twinx()
tax.plot(sorted_c[:, -2])
# ax.plot(sorted_c[:, 1], color = 'tab:orange')
ax.plot(sorted_c[:, -1], color = 'tab:orange')
# ax.set_yscale('log')
ax.set_ylabel(f'# of whole texts\n{int(sorted_c[:,1].sum())} samples')
tax.set_ylabel(f'Cumulative % of unique texts\n{int(sorted_c[:,2].sum())} samples')
# tax.set_yticks([0, 25, 50, 75, 100])
ax.set_xlabel('Cluster')

thr_cs = np.where(sorted_c[:, -2] >= thr)[0][0]
tax.axvline(thr_cs, color = 'tab:red', linestyle = '--')

# sorted_c의 thr_cs번째(-2, -1) 텍스트값을 뽑아서, axvline에 겹치지 않게 표시
from matplotlib.ticker import FuncFormatter

# thr_cs에 해당하는 x 좌표
x_val = thr_cs
y_val_right = sorted_c[thr_cs, -2]
y_val_left = sorted_c[thr_cs, -1]

# 누적 unique %와 누적 전체 % 값 포맷
text_right = f'{y_val_right:.1f}%'
text_left = f'{y_val_left:.1f}%'

# 겹치지 않도록 x_offset, y_offset 조정
ax.annotate(
    text_left, 
    xy=(x_val, y_val_left), 
    xytext=(x_val+10, y_val_left-5),
    textcoords='data',
    arrowprops=dict(arrowstyle='->', color='tab:orange'),
    fontsize=10, color='tab:orange',
    va='bottom')

tax.annotate(
    text_right, 
    xy=(x_val, y_val_right), 
    xytext=(x_val+10, y_val_right-5),
    textcoords='data',
    arrowprops=dict(arrowstyle='->', color='tab:blue'),
    fontsize=10, color='tab:blue',
    va='bottom')

ax.legend(
    handles = [
        plt.Line2D([0], [0], color='tab:blue', lw=2, label = 'Unique'),
        plt.Line2D([0], [0], color='tab:orange', lw=2, label = 'All'),
    ],
    # labels = ['Unique', 'All'], 
    loc = 'lower right'
)

In [ ]:
merged_df.drop_duplicates(subset=['검사결과결론내용']).label.value_counts()

In [ ]:
temp_df.c.value_counts().index[0]

In [ ]:
temp_df.query('c == 925')['검사결과결론내용'].value_counts()

In [ ]:
groups = sorted_c[thr_cs:, 0].astype(int)

In [ ]:
most_c = temp_df.c.value_counts().index[0]
# neg_temp_df = temp_df.query(f'c != {most_c}').drop_duplicates(subset=['검사결과결론내용']).reset_index().drop(columns = ['c'])
# test_df = temp_df.query(f'c == {most_c}').drop_duplicates(subset=['검사결과결론내용']).reset_index().drop(columns = ['c'])
neg_temp_df = temp_df.loc[temp_df.c.isin(groups)].reset_index().drop(columns = ['c'])
test_df = temp_df.loc[~temp_df.c.isin(groups)].reset_index().drop(columns = ['c'])

In [ ]:
neg_temp_df.label.value_counts(), test_df.label.value_counts()

In [ ]:
novel_case = []
normal_case = []
for t, sdf in reviewed_df.groupby('검사결과결론내용'):
    if sdf.shape[0] == 1:
        normal_case.append(sdf)
        continue

    if sdf.label.unique().shape[0] > 1:
        # print(t)
        novel_case.append(sdf)
    else:
        normal_case.append(sdf)
    # break

In [ ]:
normal_df = pd.concat(normal_case).sort_values('index').drop_duplicates(['검사결과결론내용'])
normal_df.label.value_counts()

In [ ]:
neg_temp_df['검사결과결론내용'].drop_duplicates().value_counts()

In [ ]:
train_df = pd.concat([normal_df, neg_temp_df.drop_duplicates('검사결과결론내용')]).sort_values('index').reset_index(drop = True)

In [ ]:
train_df['검사결과결론내용'].value_counts()

In [ ]:
output_dir

In [ ]:
train_texts = train_df['검사결과결론내용'].to_list()

In [ ]:
merged_df.loc[~(merged_df['검사결과결론내용'].isin(train_texts))].drop_duplicates(subset=['검사결과결론내용']).label.value_counts()

In [ ]:
merged_df.loc[~(merged_df['검사결과결론내용'].isin(train_texts))].label.value_counts()

In [ ]:
test_df.label.value_counts()

In [ ]:
output_dir

In [ ]:
os.makedirs(output_dir, exist_ok=True)

train_df.reset_index(drop = True).to_csv(os.path.join(output_dir, f'labeled_recur.csv'), index = False, encoding = 'utf-8-sig')
test_df.reset_index(drop = True).to_csv(os.path.join(output_dir, 'non_labeled_recur.csv'), index = False, encoding = 'utf-8-sig')